<a href="https://colab.research.google.com/github/fa7e/Calculador-de-Stock/blob/main/reposicion09_05_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from google.colab import files
from datetime import datetime

# ==========================================
# SUBIR ARCHIVO
# ==========================================

uploaded = files.upload()

file_path = next(iter(uploaded))
df = pd.read_excel(file_path, dtype={'code': str})

# ==========================================
# LIMPIAR DATOS
# ==========================================

df = df.fillna(0)

for col in df.columns[:2]:
    df[col] = df[col].astype(str)

for col in df.columns[2:]:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# ==========================================
# FILTRAR PRODUCTOS CON STOCK EN CENTRAL
# ==========================================

df = df[df['Central'] > 0]

# ==========================================
# CREAR COLUMNAS DE REPOSICIÓN
# ==========================================

reposiciones = [
    'reposicion Apoquindo',
    'reposicion Concepcion',
    'reposicion Costanera',
    'reposicion Calama',
    'reposicion Viña',
    'reposicion Kennedy',
    'reposicion Dominicos',
    'reposicion MKT',
    'reposicion Web'
]

for col in reposiciones:
    df[col] = 0

df['quiebre'] = False

# ==========================================
# SKU RESTRINGIDOS
# ==========================================

sku_restringidos = {
    "1000009090","1000009091","1000009092","1000009093","1000009094",
    "1000009103","1000009104","1000009106","1000009122","1000009123",
    "1000009124","1000009125","1000009126","1000009127","1000009128",
    "1000009129","1000009130","1000009131","1000009132","1000009133",
    "1000009134","1000009135","1000009136","9201010107","9201020101",
    "9201020105","9201030104","9201030106","9202040101","9202040107",
    "9202050106","9202050108","9203060100","9203070101","1000009105",
    "1000009121"
}

# ==========================================
# BODEGAS PERMITIDAS
# ==========================================

bodegas_permitidas = {
    'reposicion Web',
    'reposicion MKT',
    'reposicion Costanera',
    'reposicion Dominicos',
    'reposicion Apoquindo'
}

# ==========================================
# FUNCIÓN DE REPOSICIÓN
# ==========================================

def calcular_reposicion(row, venta_media_col, stock_col, reposicion_col):

    codigo = row['code']

    # Validar stock central
    if row['Central'] <= 0:
        row['quiebre'] = True
        return row

    # Validar restricción de SKU
    if codigo in sku_restringidos:
        if reposicion_col not in bodegas_permitidas:
            return row

    # Stock mínimo protegido
    stock_central = max(row['Central'], 0)
    stock_minimo = int(stock_central * 0.3)
    stock_disponible = stock_central - stock_minimo

    if stock_disponible <= 0:
        row['quiebre'] = True
        return row

    # Lógica de reposición
    if row[venta_media_col] > 0 and row[stock_col] == 0:
        cantidad = min(row[venta_media_col], stock_disponible)

    elif row[venta_media_col] == 0 and row[stock_col] == 0:
        cantidad = min(1, stock_disponible)

    elif row[venta_media_col] > row[stock_col]:
        cantidad = min(
            row[venta_media_col] - row[stock_col],
            stock_disponible
        )

    else:
        cantidad = 0

    # Aplicar reposición
    row[reposicion_col] += cantidad
    row['Central'] -= cantidad

    # Validar quiebre
    if row['Central'] <= stock_minimo:
        row['quiebre'] = True

    return row

# ==========================================
# SUCURSALES DISPONIBLES
# ==========================================

bodegas_disponibles = [
    (1, 'MKT', 'venta mensual MKT', 'stock MKT', 'reposicion MKT'),
    (2, 'Apoquindo', 'venta mensual Apoquindo', 'stock Apoquindo', 'reposicion Apoquindo'),
    (3, 'Concepcion', 'venta mensual Concepcion', 'stock  Concepcion', 'reposicion Concepcion'),
    (4, 'Costanera', 'venta mensual Costanera', 'stock  Costanera', 'reposicion Costanera'),
    (5, 'Calama', 'venta mensual Dji-Calama', 'stock  Dji-Calama', 'reposicion Calama'),
    (6, 'Viña', 'venta mensual Dji-Viña', 'stock  Dji-Viña', 'reposicion Viña'),
    (7, 'Kennedy', 'venta mensual Kennedy', 'stock  Kennedy', 'reposicion Kennedy'),
    (8, 'Dominicos', 'venta mensual Dominicos', 'stock Dominicos', 'reposicion Dominicos'),
    (9, 'Web', 'venta mensual Web', 'stock Web', 'reposicion Web'),
]

# ==========================================
# MOSTRAR SUCURSALES
# ==========================================

print("\n==========================================")
print("SUCURSALES DISPONIBLES")
print("==========================================\n")

for numero, nombre, _, _, _ in bodegas_disponibles:
    print(f"{numero}. {nombre}")

print("\nOpciones:")
print(" - Escribe 'T' para cargar TODAS")
print(" - O escribe números separados por coma")
print("   Ejemplo: 1,3,5\n")

seleccion = input("Seleccione sucursales: ").strip()

# ==========================================
# FILTRAR SUCURSALES SELECCIONADAS
# ==========================================

if seleccion.upper() == 'T':

    bodegas = [
        (venta, stock, repo)
        for _, _, venta, stock, repo in bodegas_disponibles
    ]

else:

    numeros = [
        int(x.strip())
        for x in seleccion.split(',')
        if x.strip().isdigit()
    ]

    bodegas = [
        (venta, stock, repo)
        for numero, _, venta, stock, repo in bodegas_disponibles
        if numero in numeros
    ]

# ==========================================
# VALIDACIÓN
# ==========================================

if len(bodegas) == 0:
    raise Exception("No se seleccionaron sucursales válidas")

# ==========================================
# MOSTRAR SUCURSALES ELEGIDAS
# ==========================================

print("\n==========================================")
print("SUCURSALES SELECCIONADAS")
print("==========================================")

for venta, stock, repo in bodegas:
    print(f"- {repo}")

# ==========================================
# APLICAR REPOSICIÓN
# ==========================================

for venta_media_col, stock_col, reposicion_col in bodegas:

    print(f"\nProcesando {reposicion_col}...")

    df = df.apply(
        lambda row: calcular_reposicion(
            row,
            venta_media_col,
            stock_col,
            reposicion_col
        ),
        axis=1
    )

# ==========================================
# COLUMNAS EXPORTADAS
# SOLO SUCURSALES SELECCIONADAS
# ==========================================

reposiciones_seleccionadas = [
    repo for _, _, repo in bodegas
]

columns_to_export = [
    'code',
    'description',
    'Central',
    'quiebre'
] + reposiciones_seleccionadas

df_export = df[columns_to_export]

# ==========================================
# FILTRAR SOLO DONDE HAYA REPOSICIÓN
# ==========================================

df_export = df_export[
    (df_export[reposiciones_seleccionadas] > 0).any(axis=1)
]

# ==========================================
# BLACKLIST GENERAL
# ==========================================

lista_negra = [
    "1000003766", "1000002714", "1000002576", "1000002431", "1000001962",
    "1000001950", "1000002680", "1000002858", "1000002466", "1000002868",
    "1000003570", "1000001190", "1000002802", "1000002602", "1000002700",
    "1000001876", "1000001454", "1000003723", "1000003734", "1000003147",
    "1000003198", "1000001684", "1000001418", "1000002614", "1000003763",
    "1000002656", "1000002429", "1000002253", "1000002425", "1000002664",
    "1000003736", "1000002272", "1000002461", "1000001867", "1000002715",
    "1000001868", "1000001877", "1000001441", "1000001422", "1000000173",
    "1000002838", "1000001679", "1000003009", "1000001419", "1000001680",
    "1000001421", "1000002346", "1000001683", "1000001869", "1000001423",
    "1000001686", "1000001542", "1000002888", "1000001006", "1000001007",
    "1000001685", "1000001005", "1000000500", "1000001678", "1000003733",
    "1000003663", "1000000951", "1000002066", "1000001008", "1000001914",
    "1000001143", "1000002573", "1000002610", "1000002607", "1000000914",
    "1000001568", "1000001870", "1000002650", "1000002605", "1000002671",
    "1000002859", "1000002611", "1000001609", "1000003917", "1000003902",
    "1000003915", "1000003908", "1000003903", "1000002010", "1000001191",
    "1000003901", "1000003765", "1000003920", "1000003912", "1000003900",
    "1000004471", "1000002612", "1000002853", "1000006927", "1000004551",
    "1000006387", "1000004466", "1000002020", "1000002623", "1000003068",
    "1000003715", "1000003728", "1000003729", "1000003068","1000005085",
    "1000005086", "1000005204", "1000005234",
    "1000003564", "1000003583", "1000002758", "1000002905", "1000003584",
    "1000004459", "1000008018", "1000008024", "1000008581", "1000006743",
    "1000002029", "1000002613", "1000002019"
]

lista_negra = [
    str(x).strip().lstrip('0')
    for x in lista_negra
]

# ==========================================
# LIMPIAR CÓDIGOS
# ==========================================

df_export['code'] = (
    df_export['code']
    .astype(str)
    .str.strip()
    .str.replace(r'\D', '', regex=True)
    .str.lstrip('0')
)

# ==========================================
# FILTRAR BLACKLIST
# ==========================================

df_filtrado = df_export[
    ~df_export['code'].isin(lista_negra)
]

# ==========================================
# RESULTADOS
# ==========================================

print("\n==========================================")
print("RESULTADOS")
print("==========================================")

print("Registros originales:", len(df_export))
print("Registros finales:", len(df_filtrado))

# ==========================================
# EXPORTAR
# ==========================================

fecha_actual = datetime.now().strftime('%d-%m-%Y')

output_file_path = f'STOCK sugerido {fecha_actual}.xlsx'

df_filtrado.to_excel(output_file_path, index=False)

print(f"\nArchivo generado: {output_file_path}")

files.download(output_file_path)

Saving STOCK PARA TIENDAS - copia.xlsx to STOCK PARA TIENDAS - copia (1).xlsx

SUCURSALES DISPONIBLES

1. MKT
2. Apoquindo
3. Concepcion
4. Costanera
5. Calama
6. Viña
7. Kennedy
8. Dominicos
9. Web

Opciones:
 - Escribe 'T' para cargar TODAS
 - O escribe números separados por coma
   Ejemplo: 1,3,5

Seleccione sucursales: 5,6

SUCURSALES SELECCIONADAS
- reposicion Calama
- reposicion Viña

Procesando reposicion Calama...

Procesando reposicion Viña...

RESULTADOS
Registros originales: 144
Registros finales: 116

Archivo generado: STOCK sugerido 09-05-2026.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>